**Bronze Layer (Raw Ingestion)**

In [0]:
from pyspark.sql.functions import *

#Create path variables

base_path = "/Volumes/workspace/ecommerce/ecommerce_data"
bronze_path = f"{base_path}/delta/bronze_events"
silver_path = f"{base_path}/delta/silver_events"
gold_path = f"{base_path}/delta/gold_product"

# 1. READ RAW DATA - using both Oct and Nov data

df_raw = spark.read.option("header", "true") \
                    .option("inferSchema", "true") \
                    .csv(f"{base_path}/2019-*.csv")

In [0]:
#2. Add Audit Columns for bronze
df_bronze = df_raw.withColumn("ingestion_ts", current_timestamp())

#3.Write to Bronze (Append Only for Bronze)
df_bronze.write.format("delta").mode("append").save(bronze_path)

print(f"Bronze Layer successfully built at: {bronze_path}")


**Silver Layer (Cleaning & Enrichment)**

In [0]:
# 1. READ FROM BRONZE
df_bronze_src = spark.read.format("delta").load(bronze_path)

# 2. Cleanup Data
# Minimal fix: cast price to float before filtering
from pyspark.sql.functions import col

df_bronze_cleaned = df_bronze_src \
    .filter(col("price").cast("float") > 0) \
    .filter(col("price").cast("float") < 10000) \
    .dropDuplicates(["user_session", "event_time", "product_id"])

# 3. Enrichment (Derived Columns)
# - Add Date Column for easier partitioning / querying
# - Add 'price_tier' for segmentation Analysis

df_silver = df_bronze_cleaned \
    .withColumn("event_date", to_date(col("event_time"))) \
    .withColumn("price_tier", 
        when(col("price").cast("float") < 50 , "Cheap")
        .when((col("price").cast("float") >= 50) & (col("price").cast("float") < 300), "Standard")
        .otherwise("Luxury")
    )
        
# 4. Write to Silver
df_silver.write.format("delta").mode("overwrite").save(silver_path)

print(f"Silver Layer successfully built at: {silver_path}")

# 5. Validation
df_silver.select("event_time", "event_type", "product_id", "price", "price_tier", "event_date").show(5)

**The Gold Layer (Business Aggregates)**

In [0]:
#1. Read from Silver
df_silver_src = spark.read.format("delta").load(silver_path)

# 2. Aggregations - Counts vies vs purchase per product

df_gold = df_silver_src.groupBy("product_id", "category_code", "brand") \
    .agg(
        #Count unique users who viewed
        countDistinct(when(col("event_type") == "view", col("user_id"))).alias("unique_views"),
        #Count Unique users who purchased
        countDistinct(when(col("event_type") == "purchase", col("user_id"))).alias("unique_purchases"),
        #Total Revenue
        sum(when(col("event_type") == "purchase", col("price").cast("float"))).alias("revenue")
      )
    
# 3. Add KPIs(Conversion Rate)
# Purchases /Views 
df_gold_final = df_gold.withColumn(
  "conversion_rate_pct", (col("unique_purchases") / (col("unique_views") + 1)) *100
).fillna(0)

# 4. Write to Gold
df_gold_final.write.format("delta").mode("overwrite").save(gold_path)
print(f"Gold Layer successfully built at: {gold_path}")

# 5. Validation
df_gold_final.select("product_id", "category_code", "brand", "unique_views", "unique_purchases", "revenue", "conversion_rate_pct").show(5)
    


**Visualization (Business Value)**

In [0]:
# Load gold Data
gold_data = spark.read.format("delta").load(gold_path)

#Visualize Top Performing Products by Revenue
display(gold_data.orderBy(col("revenue").desc()).limit(10))